# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashu141d/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
import os
import itertools
import numpy as np
import pandas as pd
from google.colab import userdata
from datasets import load_dataset

# 1. Authenticate and stream warehouse slice
hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train",
    token=hf_token
)

# Pull 60,000 daily rows to construct clean temporal/grouped page observations
raw_df = pd.DataFrame(list(itertools.islice(ds, 60000)))
cols = raw_df.columns.tolist()

key_col = next((c for c in cols if any(k in c.lower() for k in ["page", "content", "url", "slug", "hash"]) and "client" not in c.lower()), cols[0])
group_col = next((c for c in cols if "client" in c.lower()), key_col)
date_col = next((c for c in cols if "date" in c.lower() or "month" in c.lower() or "day" in c.lower()), None)
imp_col = next((c for c in cols if "imp" in c.lower()), None)
clk_col = next((c for c in cols if "click" in c.lower()), None)
pos_col = next((c for c in cols if "pos" in c.lower()), None)

print(f"Loaded {len(raw_df)} rows | Entity key: {key_col} | Group key: {group_col} | Date col: {date_col}")

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loaded 60000 rows | Entity key: content_hash_id | Group key: client_hash_id | Date col: report_date


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
from sklearn.model_selection import GroupKFold

# Build leak-free observation features (window 1) -> future outcome target (window 2)
if date_col is not None:
    raw_df["_dt"] = pd.to_datetime(raw_df[date_col], errors="coerce")
    valid_dt = raw_df.dropna(subset=["_dt"]).copy()
    cutoff_dt = valid_dt["_dt"].quantile(0.50)

    obs_df = valid_dt[valid_dt["_dt"] <= cutoff_dt]
    fut_df = valid_dt[valid_dt["_dt"] > cutoff_dt]

    agg_obs = {imp_col: ["sum", "std"], clk_col: "sum", group_col: "first"}
    if pos_col:
        agg_obs[pos_col] = ["mean", "std"]

    feat_df = obs_df.groupby(key_col).agg(agg_obs)
    feat_df.columns = ["_".join(col).strip() for col in feat_df.columns.values]
    feat_df = feat_df.reset_index()

    fut_agg = fut_df.groupby(key_col, as_index=False).agg(future_clicks=(clk_col, "sum"), future_imps=(imp_col, "sum"))
    model_df = feat_df.merge(fut_agg, on=key_col, how="inner")

    imp_sum_col = f"{imp_col}_sum"
    clk_sum_col = f"{clk_col}_sum"
    grp_clean_col = f"{group_col}_first"

    model_df = model_df[model_df[imp_sum_col] >= 20].copy()
    # Target: click rate / volume drop >= 20% in the subsequent window
    obs_rate = model_df[clk_sum_col] / model_df[imp_sum_col].clip(lower=1)
    fut_rate = model_df["future_clicks"] / model_df["future_imps"].clip(lower=1)
    model_df["target"] = ((fut_rate - obs_rate) / obs_rate.clip(lower=1e-4) <= -0.20).astype(int)
else:
    # Fallback page-level aggregation if no date column exists
    agg_dict = {imp_col: "sum", clk_col: "sum", group_col: "first"}
    if pos_col:
        agg_dict[pos_col] = "mean"
    model_df = raw_df.groupby(key_col, as_index=False).agg(agg_dict)
    imp_sum_col, clk_sum_col, grp_clean_col = imp_col, clk_col, group_col
    model_df = model_df[model_df[imp_sum_col] >= 20].copy()
    ctr_tmp = model_df[clk_sum_col] / model_df[imp_sum_col].clip(lower=1)
    model_df["target"] = (ctr_tmp < ctr_tmp.median()).astype(int)

# Strictly decision-time features (NO future_* columns)
model_df["log_impressions"] = np.log1p(model_df[imp_sum_col])
model_df["ctr_obs"] = model_df[clk_sum_col] / model_df[imp_sum_col].clip(lower=1)
feature_names = ["log_impressions", "ctr_obs"]

if f"{imp_col}_std" in model_df.columns:
    model_df["imp_volatility"] = model_df[f"{imp_col}_std"].fillna(0.0)
    feature_names.append("imp_volatility")
if pos_col and f"{pos_col}_mean" in model_df.columns:
    model_df["avg_position"] = model_df[f"{pos_col}_mean"].fillna(model_df[f"{pos_col}_mean"].median())
    model_df["pos_volatility"] = model_df[f"{pos_col}_std"].fillna(0.0)
    feature_names.extend(["avg_position", "pos_volatility"])
elif pos_col and pos_col in model_df.columns:
    model_df["avg_position"] = model_df[pos_col].fillna(model_df[pos_col].median())
    feature_names.append("avg_position")

X = model_df[feature_names].copy()
y = model_df["target"].values
groups = model_df[grp_clean_col].astype(str).values

n_splits = min(5, len(np.unique(groups)))
gkf = GroupKFold(n_splits=max(2, n_splits))

print(f"Modeling dataset shape: {X.shape} | Positive target rate: {y.mean():.3f}")
print(f"Features used (strictly decision-time): {feature_names}")
print(f"Validation design: GroupKFold(n_splits={max(2, n_splits)}) grouped by '{grp_clean_col}'")

Modeling dataset shape: (3208, 5) | Positive target rate: 0.224
Features used (strictly decision-time): ['log_impressions', 'ctr_obs', 'imp_volatility', 'avg_position', 'pos_volatility']
Validation design: GroupKFold(n_splits=3) grouped by 'client_hash_id_first'


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

def precision_at_k(y_true, y_score, k=25):
    k = min(k, len(y_true))
    top_idx = np.argsort(y_score)[::-1][:k]
    return float(np.mean(y_true[top_idx]))

# Out-of-fold predictions on identical GroupKFold splits
oof_baseline = np.zeros(len(y))
oof_logreg = np.zeros(len(y))
oof_rf = np.zeros(len(y))

for train_idx, val_idx in gkf.split(X, y, groups):
    X_tr, y_tr = X.iloc[train_idx], y[train_idx]
    X_va, y_va = X.iloc[val_idx], y[val_idx]

    # 1. Week-4 Heuristic Baseline (fit median CTR on train fold only)
    med_ctr_tr = X_tr["ctr_obs"].median()
    if "avg_position" in X_va.columns:
        in_band = (X_va["avg_position"] >= 3.0) & (X_va["avg_position"] <= 12.0)
    else:
        in_band = pd.Series(True, index=X_va.index)
    ctr_gap = (med_ctr_tr - X_va["ctr_obs"]).clip(lower=0.0)
    oof_baseline[val_idx] = np.where(in_band, np.expm1(X_va["log_impressions"]) * ctr_gap, 0.0)

    # 2. Logistic Regression
    lr_pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
    lr_pipe.fit(X_tr, y_tr)
    oof_logreg[val_idx] = lr_pipe.predict_proba(X_va)[:, 1]

    # 3. Regularized Random Forest
    rf_clf = RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=15, random_state=42, n_jobs=-1)
    rf_clf.fit(X_tr, y_tr)
    oof_rf[val_idx] = rf_clf.predict_proba(X_va)[:, 1]

results_table = pd.DataFrame([
    {
        "Model / Baseline": "Week-4 Heuristic Rule (Baseline)",
        "PR-AUC (Avg Precision)": round(average_precision_score(y, oof_baseline), 4),
        "ROC-AUC": round(roc_auc_score(y, oof_baseline), 4),
        "Precision@25": round(precision_at_k(y, oof_baseline, k=25), 4)
    },
    {
        "Model / Baseline": "Logistic Regression (L2 + Scaled)",
        "PR-AUC (Avg Precision)": round(average_precision_score(y, oof_logreg), 4),
        "ROC-AUC": round(roc_auc_score(y, oof_logreg), 4),
        "Precision@25": round(precision_at_k(y, oof_logreg, k=25), 4)
    },
    {
        "Model / Baseline": "Random Forest (max_depth=6, leaf=15)",
        "PR-AUC (Avg Precision)": round(average_precision_score(y, oof_rf), 4),
        "ROC-AUC": round(roc_auc_score(y, oof_rf), 4),
        "Precision@25": round(precision_at_k(y, oof_rf, k=25), 4)
    }
])

print("=== Model-vs-Baseline Comparison (Out-of-Fold GroupKFold) ===")
display(results_table)

=== Model-vs-Baseline Comparison (Out-of-Fold GroupKFold) ===


,Model / Baseline,PR-AUC (Avg Precision),ROC-AUC,Precision@25
0,Week-4 Heuristic Rule (Baseline),0.2244,0.5000,0.12
1,Logistic Regression (L2 + Scaled),0.7494,0.9436,0.80
2,"Random Forest (max_depth=6, leaf=15)",0.7868,0.9538,0.92


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
from sklearn.inspection import permutation_importance

# Fit on final fold to inspect permutation importance and concrete error cases
train_idx, val_idx = list(gkf.split(X, y, groups))[-1]
X_tr, y_tr = X.iloc[train_idx], y[train_idx]
X_va, y_va = X.iloc[val_idx], y[val_idx]

final_rf = RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=15, random_state=42, n_jobs=-1)
final_rf.fit(X_tr, y_tr)

perm = permutation_importance(final_rf, X_va, y_va, scoring="average_precision", n_repeats=10, random_state=42)
perm_df = pd.DataFrame({
    "feature": feature_names,
    "importance_mean": perm.importances_mean.round(4),
    "importance_std": perm.importances_std.round(4)
}).sort_values(by="importance_mean", ascending=False)

print("=== Permutation Importance (Validation Fold, PR-AUC Drop) ===")
display(perm_df)

# Inspect top False Positives and False Negatives
val_audit = model_df.iloc[val_idx][[key_col] + feature_names + ["target"]].copy()
val_audit["pred_prob"] = final_rf.predict_proba(X_va)[:, 1].round(4)

print("\n--- Sample False Positives (High Predicted Risk, Actual Target = 0) ---")
display(val_audit[val_audit["target"] == 0].sort_values(by="pred_prob", ascending=False).head(3))

print("\n--- Sample False Negatives (Low Predicted Risk, Actual Target = 1) ---")
display(val_audit[val_audit["target"] == 1].sort_values(by="pred_prob", ascending=True).head(3))

=== Permutation Importance (Validation Fold, PR-AUC Drop) ===


,feature,importance_mean,importance_std
1,ctr_obs,0.5833,0.119
0,log_impressions,0.0000,0.000
2,imp_volatility,0.0000,0.000
3,avg_position,0.0000,0.000
4,pos_volatility,0.0000,0.000



--- Sample False Positives (High Predicted Risk, Actual Target = 0) ---


,content_hash_id,log_impressions,ctr_obs,imp_volatility,avg_position,pos_volatility,target,pred_prob
1131,content_3fcbde3da0190897,5.631212,0.0,11.910821,831.823529,589.959981,0,0.0147
2436,content_862904aa99cb45d6,3.988984,0.0,15.628500,107.000000,108.268186,0,0.0138
2305,content_7eefb47fe5db7831,3.871201,0.0,23.334524,1214.500000,1147.634306,0,0.0056



--- Sample False Negatives (Low Predicted Risk, Actual Target = 1) ---


,content_hash_id,log_impressions,ctr_obs,imp_volatility,avg_position,pos_volatility,target,pred_prob
4246,content_e5acd18462ddd441,4.26268,0.028571,6.610177,122.333333,103.164917,1,0.7266
2659,content_913f91b1e9713a10,3.76120,0.023810,3.609105,215.153846,244.424783,1,0.8317


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.